# Logistic regression
# Data Cleaning and Preprocessing

## Data Cleaning

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.feature_selection import SelectKBest, f_classif
import psycopg2
from sqlalchemy import create_engine
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt

In [ ]:
# Define the connection parameters
db_params = {
    'host': '194.171.191.226',
    'port': '6379',  # Assuming 6379 is correct
    'database': 'postgres',
    'user': 'group2',
    'password': 'blockd_2024group2_58'
}

# Create a connection string
conn_string = f"postgresql+psycopg2://{db_params['user']}:{db_params['password']}@{db_params['host']}:{db_params['port']}/{db_params['database']}"

# Create an SQLAlchemy engine
engine = create_engine(conn_string)

# Use pandas to execute a query and load data into a DataFrame
query = "SELECT * FROM data_lake.safe_driving"
df_safe_driving = pd.read_sql(query, engine)

# Close the connection
engine.dispose()

# Ensure 'event_start' and 'event_end' are in datetime format
df_safe_driving['event_start'] = pd.to_datetime(df_safe_driving['event_start'])
df_safe_driving['event_end'] = pd.to_datetime(df_safe_driving['event_end'])

# Check for duplicate values
dup_count = df_safe_driving.duplicated().sum()
print('There are', dup_count, 'duplicate values')

# Re-check for missing values in different forms
missing_count = df_safe_driving.isnull().sum().sum()
print('There are', missing_count, 'missing values')

# Replace empty strings with NaN in the 'road_number' column
df_safe_driving['road_number'] = df_safe_driving['road_number'].replace("", np.nan)

# Verify that the replacement has been done
missing_values_count = df_safe_driving.isnull().sum()
print("Total missing values per column:")
print(missing_values_count[missing_values_count > 0])

# Summarize findings
total_missing = missing_values_count.sum()
print(f"Total number of missing values (including special values): {total_missing}")

# Verify that the rows with missing values
print(df_safe_driving.isna().sum())

# Print start cleaning
print('\nStarting Cleaning:\n')

# Remove the 'road_number' column
df_safe_driving = df_safe_driving.drop('road_number', axis=1)

# Verify that the rows with missing 'road_number' values have been removed
print(df_safe_driving.isna().sum())

# Strip whitespace from the 'category' column
df_safe_driving['category'] = df_safe_driving['category'].str.strip()

# Print the unique values
print(df_safe_driving['category'].unique())

# Check for duplicate values
dup_count = df_safe_driving.duplicated().sum()
print('After cleaning there are', dup_count, 'duplicate values')

# Re-check for missing values in different forms
missing_count = df_safe_driving.isnull().sum().sum()
print('After cleaning there are', missing_count, 'missing values')

# Display the first few rows of the cleaned DataFrame
df_safe_driving.head()

## Cleaning outliers

In [ ]:
# Count the number of rows before removing outliers
rows_before = df_safe_driving.shape[0]

# Define the coordinates of the outliers
outlier_coords = [(51.605330, 4.779188), (51.590760, 4.811617)]

# Remove rows for outliers
for coord in outlier_coords:
    df_safe_driving = df_safe_driving[~((df_safe_driving['latitude'] == coord[0]) & (df_safe_driving['longitude'] == coord[1]))]

# Find the index of the outlier
outlier_index = df_safe_driving[df_safe_driving['duration_seconds'] > 500].index

# Drop the outlier row from the DataFrame
df_safe_driving = df_safe_driving.drop(outlier_index)

# Count the number of rows after removing outliers
rows_after = df_safe_driving.shape[0]

# Calculate the number of rows removed
rows_removed = rows_before - rows_after

# Print the number of rows removed
print("Number of rows removed:", rows_removed)

## Creating bins

In [ ]:
# Initialize LabelEncoder
label_encoder = LabelEncoder()

# Encode the 'category' column
df_safe_driving['category_encoded'] = label_encoder.fit_transform(df_safe_driving['category'])

# Strip whitespace from the 'category' column
df_safe_driving['incident_severity'] = df_safe_driving['incident_severity'].str.strip()

# Print the unique values
print(df_safe_driving['incident_severity'].unique())

def classify_incident_severity_corrected(severity):
    low = ['HA1', 'HB1', 'HC1', 'SP1']
    medium = ['HC4', 'HC13', 'HA2', 'HB2', 'HC2', 'HC5', 'HC7', 'HC14', 'HC16', 'SP2', 'SP3']
    high = ['HA3', 'HB3', 'HC3', 'HC9', 'HC11', 'HC12', 'HC15', 'HC17', 'HC18', 'HC19', 'HC20', 'HC21', 'HC6', 'HC8', 'HC10', 'SP4', 'SP5']
    
    if severity in low:
        return 'Low'
    elif severity in medium:
        return 'Medium'
    elif severity in high:
        return 'High'
    else:
        return 'Unknown'

# Apply the corrected classification function
df_safe_driving['incident_severity_bin'] = df_safe_driving['incident_severity'].apply(classify_incident_severity_corrected)

# Display the updated DataFrame
df_safe_driving.head()

In [ ]:
# Create a count histogram for the incident_severity_bin
plt.figure(figsize=(10, 6))
df_safe_driving['incident_severity_bin'].value_counts().plot(kind='bar', color=['blue', 'orange', 'red'])
plt.title('Count of Incident Severity Bins')
plt.xlabel('Incident Severity Bin')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.show()

## Resampling the bins

In [ ]:
# Apply Random Sampling with Replacement
sampled_dfs = []
for bin_label, group in df_safe_driving.groupby('incident_severity_bin'):
    sampled_dfs.append(group.sample(n=63533, replace=True, random_state=42))
df_sampled_replacement = pd.concat(sampled_dfs)

# Display the count for each incident severity bin in the resampled data
sampled_replacement_bin_counts = df_sampled_replacement['incident_severity_bin'].value_counts()
print("Random Sampling with Replacement bin counts:\n", sampled_replacement_bin_counts)

# Set the resampled DataFrame to df_safe_driving
df_safe_driving = df_sampled_replacement

# Plot histogram
sampled_replacement_bin_counts.plot(kind='bar', color=['skyblue', 'orange', 'green'])
plt.xlabel('Incident Severity Bin')
plt.ylabel('Count')
plt.title('Counts of Incident Severity Bins After Random Sampling with Replacement')
plt.xticks(rotation=0)
plt.show()

## Train test split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df_safe_driving[['duration_seconds', 'speed_kmh', 'end_speed_kmh', 'maxwaarde', 'category_encoded']]  # Features
y = df_safe_driving['incident_severity_bin']

# Step 2: Split the data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 3: Split the train data into train and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

# Step 4: Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)  
X_test_scaled = scaler.transform(X_test) 

### Simple logistic regression

In [ ]:
# make and fit the logistic regression model
logreg_model = LogisticRegression(max_iter=1000)  # increasing the number of iterations
logreg_model.fit(X_train_scaled, y_train)

# make it make predictions
logreg_predictions = logreg_model.predict(X_test_scaled)

# evaluation of the model
overall_accuracy = accuracy_score(y_test, logreg_predictions)

# calculating macro-averaged precision, recall, and F1-score
precision, recall, f1, _ = precision_recall_fscore_support(y_test, logreg_predictions, average='macro')

print("\nOverall Model Performance:")
print("Overall Accuracy:", overall_accuracy)
print("Macro-Averaged Precision:", precision)
print("Macro-Averaged Recall:", recall)
print("Macro-Averaged F1-score:", f1)

### Logistic regression with regularization and classification report

In [ ]:
# Select everything
X_train_selected = X_train_scaled
X_test_selected = X_test_scaled

# make and fit the logistic regression model with regularization
logreg_model_regularized = LogisticRegression(penalty='l2', C=1.0, max_iter=1000)  # L2 regularization with default regularization strength
logreg_model_regularized.fit(X_train_selected, y_train)

#predictions 
logreg_predictions_regularized = logreg_model_regularized.predict(X_test_selected)

# Evaluation
overall_accuracy_regularized = accuracy_score(y_test, logreg_predictions_regularized)

# generate scores
precision_regularized, recall_regularized, f1_regularized, _ = precision_recall_fscore_support(y_test, logreg_predictions_regularized, average='macro')

# printing scores
print("\nClassification Report (Regularized Logistic Regression):")
print(classification_report(y_test, logreg_predictions_regularized))

print("\nOverall Model Performance (Regularized Logistic Regression):")
print("Overall Accuracy:", overall_accuracy_regularized)
print("Macro-Averaged Precision:", precision_regularized)
print("Macro-Averaged Recall:", recall_regularized)
print("Macro-Averaged F1-score:", f1_regularized)

### Logistic regression with grid search

In [ ]:
# split the data into training and test sets with stratification to maintain class distribution
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# handle class imbalance with smote
smote = SMOTE(random_state=42)
x_train_resampled, y_train_resampled = smote.fit_resample(x_train, y_train)

# define a pipeline with feature selection, scaler, and logistic regression
pipeline = Pipeline([
    ('feature_selection', SelectKBest(score_func=f_classif)),  # feature selection
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(max_iter=1000))
])

# adjust the parameter grid to avoid the issue with k being greater than the number of features
param_grid = {
    'feature_selection__k': [2, 3],  # adjusted to be <= n_features
    'logreg__C': [0.01, 0.1, 1, 10, 100],
    'logreg__solver': ['liblinear', 'lbfgs', 'saga'],
    'logreg__penalty': ['l2', 'none'],
    'logreg__max_iter': [100, 500, 1000]
}

# perform grid search with cross-validation
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
grid_search = GridSearchCV(pipeline, param_grid, cv=cv, scoring='accuracy', n_jobs=-1, verbose=1)
grid_search.fit(x_train_resampled, y_train_resampled)

# get the best model
best_model = grid_search.best_estimator_

# make predictions
logreg_predictions = best_model.predict(x_test)

# evaluate the model
overall_accuracy = accuracy_score(y_test, logreg_predictions)
precision, recall, f1, _ = precision_recall_fscore_support(y_test, logreg_predictions, average='macro')

# print classification report
print("\nClassification Report (Regularized Logistic Regression):")
print(classification_report(y_test, logreg_predictions))

print("\nOverall Model Performance (Regularized Logistic Regression):")
print("Overall Accuracy:", overall_accuracy)
print("Macro-Averaged Precision:", precision)
print("Macro-Averaged Recall:", recall)
print("Macro-Averaged F1-score:", f1)

# print confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, logreg_predictions))

# print the best hyperparameters
print("\nBest Hyperparameters:")
print(grid_search.best_params_)

### Using now a random forest classifier with grid search

In [ ]:
# Initialize the Random Forest model
rf_model = RandomForestClassifier(random_state=42)

# Define the parameter grid for Random Forest
param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Initialize GridSearchCV for Random Forest
grid_search_rf = GridSearchCV(rf_model, param_grid_rf, cv=5, scoring='accuracy', n_jobs=-1)

# Fit the Random Forest model using GridSearchCV
grid_search_rf.fit(X_train_scaled, y_train)

# Get the best parameters and the best model
best_params_rf = grid_search_rf.best_params_
best_model_rf = grid_search_rf.best_estimator_

print(f"Best Parameters for Random Forest: {best_params_rf}")

# Predict on the validation set using the best Random Forest model
y_val_pred_rf = best_model_rf.predict(X_val_scaled)

# Evaluate the best Random Forest model
accuracy_rf = accuracy_score(y_val, y_val_pred_rf)
precision_rf, recall_rf, fscore_rf, _ = precision_recall_fscore_support(y_val, y_val_pred_rf, average='weighted')

print(f"Validation Accuracy (RF): {accuracy_rf:.4f}")
print(f"Validation Precision (RF): {precision_rf:.4f}")
print(f"Validation Recall (RF): {recall_rf:.4f}")
print(f"Validation F1-Score (RF): {fscore_rf:.4f}")

# Print the classification report
classification_report_rf = classification_report(y_val, y_val_pred_rf)
print("Classification Report (RF):")
print(classification_report_rf)